In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import psutil
import torch
import torch.nn as nn

raiz_projeto = str(Path().resolve().parent)
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

from src.data_loader import fixar_sementes, preparar_dados_pytorch
from src.models import MLP
from src.train import treinar

try:
    from codecarbon import EmissionsTracker
    TEM_CODECARBON = True
except ImportError:
    TEM_CODECARBON = False

os.makedirs("../reports/tables", exist_ok=True)
os.makedirs("../logs", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# 1. Carregamento dos dados
fixar_sementes(42)
loaders_reg, _, _, n_feat_reg = preparar_dados_pytorch(
    caminho_csv="../data/processed/dataset_regression.csv",
    target_col="term_gpa",
    tipo="reg",
    batch_size=32,
)
loaders_cls, _, _, n_feat_cls = preparar_dados_pytorch(
    caminho_csv="../data/processed/dataset_classification.csv",
    target_col="gpa_category_encoded",
    tipo="class",
    batch_size=32,
)

In [3]:
def calcular_macs_mlp(modelo: nn.Module) -> int:
    """Calcula o total de Multiply-Accumulate Operations (MACs) para 1 amostra."""
    macs = 0
    for camada in modelo.modules():
        if isinstance(camada, nn.Linear):
            macs += camada.in_features * camada.out_features
    return int(macs)

In [4]:
def medir_eficiencia(nome: str, tarefa: str, hidden: tuple, lr: float, loaders, in_dim: int) -> dict:
    """Mede tempo de treino, latência, throughput, RAM, MACs, energia e CO2."""
    processo = psutil.Process(os.getpid())
    ram_antes_mb = processo.memory_info().rss / (1024 ** 2)

    fixar_sementes(42)
    modelo = MLP(in_dim=in_dim, out_dim=1, hidden=hidden, activation="ReLU", dropout=0.0)
    loss_fn = nn.MSELoss() if tarefa == "regressao" else nn.BCEWithLogitsLoss()
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)

    tracker = None
    if TEM_CODECARBON:
        tracker = EmissionsTracker(
            project_name=nome.replace(" ", "_"),
            output_dir="../logs",
            log_level="error",
            save_to_file=False,
        )
        tracker.start()

    t0_treino = time.perf_counter()
    hist = treinar(
        modelo=modelo,
        dl_tr=loaders["train"],
        dl_val=loaders["val"],
        loss_fn=loss_fn,
        otimizador=otimizador,
        n_epocas=200,
        paciencia=20,
        device=device,
    )
    tempo_treino_s = time.perf_counter() - t0_treino

    if tracker is not None:
        co2_kg = float(tracker.stop() or 0.0)
        energia_kwh = float(getattr(tracker, "_total_energy", None).kWh) if getattr(tracker, "_total_energy", None) else (0.065 * tempo_treino_s / 3600.0)
        co2_gramas = co2_kg * 1000.0 if co2_kg > 0 else energia_kwh * 85.0
        metodo_energia = "CodeCarbon"
    else:
        energia_kwh = 0.065 * (tempo_treino_s / 3600.0)
        co2_gramas = energia_kwh * 85.0
        metodo_energia = "Estimativa_TDP_65W"

    epocas_rodadas = len(hist["treino"])
    ms_por_epoca = (tempo_treino_s / max(epocas_rodadas, 1)) * 1000.0
    ram_depois_mb = processo.memory_info().rss / (1024 ** 2)
    delta_ram_mb = max(0.0, ram_depois_mb - ram_antes_mb)

    # Medição de inferência (aquecimento + 500 repetições)
    modelo.cpu().eval()
    x_bs1 = torch.randn(1, in_dim)
    x_bs32 = torch.randn(32, in_dim)

    with torch.inference_mode():
        for _ in range(30):
            _ = modelo(x_bs1)
            _ = modelo(x_bs32)

        n_reps = 500
        t0_bs1 = time.perf_counter()
        for _ in range(n_reps):
            _ = modelo(x_bs1)
        t_bs1 = (time.perf_counter() - t0_bs1) / n_reps

        t0_bs32 = time.perf_counter()
        for _ in range(n_reps):
            _ = modelo(x_bs32)
        t_bs32 = (time.perf_counter() - t0_bs32) / n_reps

    n_params = modelo.contar_parametros()
    macs = calcular_macs_mlp(modelo)

    return {
        "Modelo": nome,
        "Tarefa": tarefa,
        "Arquitetura": str(hidden),
        "Parametros": n_params,
        "MACs_amostra": macs,
        "FLOPs_amostra": 2 * macs,
        "Epocas": epocas_rodadas,
        "Tempo_Treino_s": round(tempo_treino_s, 4),
        "Tempo_Epoca_ms": round(ms_por_epoca, 3),
        "Latencia_BS1_ms": round(t_bs1 * 1000.0, 4),
        "Throughput_BS1_amostras_s": round(1.0 / t_bs1, 1),
        "Latencia_BS32_ms_amostra": round((t_bs32 / 32.0) * 1000.0, 5),
        "Throughput_BS32_amostras_s": round(32.0 / t_bs32, 1),
        "RAM_Processo_MB": round(ram_depois_mb, 2),
        "Delta_RAM_MB": round(delta_ram_mb, 2),
        "Energia_Treino_Wh": round(energia_kwh * 1000.0, 6),
        "Emissao_CO2_g": round(co2_gramas, 6),
        "Metodo_Energia": metodo_energia,
    }

In [5]:
resultados_efic = [
    medir_eficiencia("MLP Baseline (Reg)", "regressao", (32, 32), 1e-3, loaders_reg, n_feat_reg),
    medir_eficiencia("MLP Otimizada (Reg)", "regressao", (128, 128), 1e-2, loaders_reg, n_feat_reg),
    medir_eficiencia("MLP Baseline (Cls)", "classificacao", (32, 32), 1e-3, loaders_cls, n_feat_cls),
    medir_eficiencia("MLP Otimizada (Cls)", "classificacao", (128, 128), 1e-3, loaders_cls, n_feat_cls),
]

df_eficiencia = pd.DataFrame(resultados_efic)
df_eficiencia.to_csv("../reports/tables/eficiencia_computacional.csv", index=False)

print("Tabela guardada em ../reports/tables/eficiencia_computacional.csv")
display(df_eficiencia)

[codecarbon WARNING @ 22:40:07] Multiple instances of codecarbon are allowed to run at the same time.


Tabela guardada em ../reports/tables/eficiencia_computacional.csv


,Modelo,Tarefa,Arquitetura,Parametros,MACs_amostra,FLOPs_amostra,Epocas,Tempo_Treino_s,Tempo_Epoca_ms,Latencia_BS1_ms,Throughput_BS1_amostras_s,Latencia_BS32_ms_amostra,Throughput_BS32_amostras_s,RAM_Processo_MB,Delta_RAM_MB,Energia_Treino_Wh,Emissao_CO2_g,Metodo_Energia
0,MLP Baseline (Reg),regressao,"(32, 32)",1281,1216,2432,36,0.3661,10.170,0.0203,49181.9,0.00099,1008596.0,850.53,179.37,0.002817,0.000277,CodeCarbon
1,MLP Otimizada (Reg),regressao,"(128, 128)",17409,17152,34304,23,0.2551,11.091,0.0236,42445.8,0.00115,870324.5,851.89,1.35,0.002335,0.000230,CodeCarbon
2,MLP Baseline (Cls),classificacao,"(32, 32)",1281,1216,2432,42,0.4121,9.811,0.0229,43734.6,0.00094,1064309.4,852.89,1.00,0.003254,0.000320,CodeCarbon
3,MLP Otimizada (Cls),classificacao,"(128, 128)",17409,17152,34304,28,0.2876,10.271,0.0243,41115.1,0.00105,949264.3,853.44,0.54,0.002212,0.000218,CodeCarbon
